In [366]:
import pandas as pd
from pandas._libs.tslibs.offsets import CustomBusinessDay
from typing import List

###### Esse notebook é de um projeto ainda em desenvolvimento. Ele é atualizado uma vez antes de cada release. Se você clonou o repositório a partir do commit mais recente, algumas informações podem estar desatualizadas. ######

# 2. Sobre os dados

A seguir estão todas as informações necessárias sobre os indicadores em que trabalhamos.

Primeiro, algumas regras gerais que valem para todos os dados coletados.

- Por enquanto, esse projeto apenas analisa dois indicadores econômicos: o CDI e o IPCA.
- Cada planilha/tabela de dados processados é referente a um indicador específico.
- As planilhas/tabelas de cada indicador armazena multiplas de suas mensurações.
- Os dados são sempre coletados e armazenados mensalmente.
- Todos os dados são armazenados em formato decimal para facilitar cálculos posteriores. A transformação de volta para a porcentagem (e a substituição de ponto para vírgula) deve ser feita apenas nas visualizações.

## 2.1 CDI

O Certificado de Depósito Interbancário (CDI) é um título emitido por bancos e instituições financeiras para permitir empréstimos entre si, e terminar o dia com o caixa positivo. Ele também é uma referência para várias outras operações financeiras no Brasil. Ele é relevante para o investidor comum quando se considera a atrelação entre o CDI, e investimentos de renda fixa como CDBs. E também porque, em geral, um investimento que rende pelo menos tanto quanto o CDI é considerado um bom investimento.

O Certificado de Depósito Interbancário é considerado uma operação _overnight_ (ou apenas _over_), o que significa que ele tem o prazo de apenas um dia útil. O índice do CDI é o valor absoluto do CDI durante esse dia. Porém, como foi feita a escolha, nesse projeto, de apenas armazenar valores mensais. O projeto apenas trabalha com a variação do CDI (ou, o CDI acumulado), em um mês. Mas estudar o índice diário do CDI ainda é relevante para nós.

### 2.1.1 O Índice do CDI

O índice do CDI (também chamado de taxa DI over) é calculado dessa maneira:

- A B3 coleta todas as operações elegíveis realizadas naquele dia. Apenas operações _interbancárias_ (extra-grupo) entre instituições financeiras registradas na B3 são consideradas. Excluindo operações feitas em um mesmo grupo bancário.
- A taxa de juros de cada operação é calculada. A seguinte fórmula é utilizada:
    $$\large DI_{i} = \left( \left( \frac{VR_{i}}{VE_{i}} \right) ^{252} - 1 \right) \times 100$$
    Onde...
    - $i$: uma operação específica
    - $VE_{i}$: valor de emissão (o que foi emprestado no início da operação)
    - $VR_{i}$: valor de resgate (o que foi devolvido no dia seguinte)
    - $DI_{i}$: taxa anual equivalente á operação.
    - $^{252}$: 252 é o número padrão de dias úteis considerados em um ano no mercado financeiro brasileiro.
    - $\times 100$: a taxa é convertida para porcentagem.
    $DI_{i}$, na prática, é o fator de crescimento do valor emprestado em um dia útil anualizado.
- Para cada taxa, é calculado um "peso" em relação à quantidade de dinheiro que os bancos emprestaram entre si no dia referente. O peso é calculado da seguinte maneira:
    $$\large Peso_{i} = \frac{VFD_{i}}{\sum_{i=1}^{n} VFD_{i}}$$
    Onde...
    - $VFD_{i}$: valor financeiro da operação $i$.
    - $n$: número total de operações elegíveis no dia.
    - $\sum VFD_{i}$: soma de todas as $n$ operações.
- Porém, se apenas utilizarmos esse valor de peso, o índice final seria muito influenciado por casos extremos. Após os cálculos dos pesos, um ajuste estatistico é aplicado aos pesos para resolver isso.
    - Primeiro, as operações são ordenadas por suas taxas, do menor para o maior.
    - A metodologia analisa a distribuição dessas taxas e identifica as operações que estão nas extremidades da distribuição (muito abaixo, ou muito acima da média)
    - É feito um procedimento de redução gradual desses pesos, onde o peso das extremidades é redistribuido para o restante das operações. O limite máximo dessa redução é controlado por um parâmetro.
    - O resultado é um conjunto de pesos ajustados, definido como pesos finais.
- Finalmente, a taxa DI do dia é calculada como uma média ponderada das taxas individuais das operações, utilizando os pesos finais. A fórmula é a seguinte:
    $$\large DI_{dia} = \sum_{i=1}^{n} \left( PesoFinal_{i} \times DI_{i} \right)$$
    Onde...
    - $DI_{i}$: taxa DI equivalente da operação
    - $PesoFinal_{i}$: peso ajustado dessa operação
    - $n$: o número total de operações elegíveis no dia

Assim, a taxa DI do dia, que resulta desse processo, funciona como uma medida representativa do custo médio de empréstimos interbancários de curtíssimo prazo no mercado brasileiro. Em resumo, aqui é importante lembrar que o CDI diário é, essencialmente, uma média ponderada das taxas de operações interbancárias naquele dia.

### 2.1.2 O CDI Armazenado e Suas Mensurações

Agora, que entendemos a fonte da taxa do CDI, e como ela é calculada, vamos falar sobre as mensurações que armazenamos no projeto. Esse é o esquema da tabela do CDI.

In [367]:
# Você pode consultar ss esquemas das planilhas em src/storage/processed/schema.py
def cdi_schema() -> pd.DataFrame:
    df = pd.DataFrame({
        "date": [],
        "cdi_annual_rate": [],
        "cdi_monthly_rate": [],
        "cdi_accumulated_ytd_rate": [],
        "cdi_12m_rate": [],
    })
    return df

Aqui, armazenamos a taxa anual (a.a) e a taxa mensal. Que são os acumulos da taxa DI Over em um ano e em um mês. No notebook 03 falaremos sobre um cuidado importante que deveremos tomar por causa da escolha de utilizar a taxa mensal como o valor principal do CDI. Por enquanto, vamos rodar a Pipeline e ver os resultados.

Utilizaremos esse comando no terminal:
```bash
py main.py --mode yearly --year 2024 --clear-data
```

Isso vai popular a planilha em /data/processed, mas note que dados brutos também são armazenados em /data/raw.
Vamos abrir o json dos dados brutos para a taxa mensal do CDI em Janeiro de 2024:
```json
{
  "reference_date": "2024-01",
  "type": "cdi",
  "value": {
    "monthly": 0.0097,
    "yearly": 0.1165
  }
}
```
Lembrando que 0.0097 equivale a 0,97%.

Bem, mas afinal, como que chegamos nesses números?
De novo, o índice do CDI se refere a apenas um dia. A taxa mensal é a mensuração do crescimento relativo do CDI entre o último dia útil do mês anterior, e o último dia útil do mês calculado. Nesse cálculo, a seguinte fórmula é utilizada.

$$\large\frac{Índice_{final}}{Índice_{inicial}} - 1$$

Verificando a série histórica do DI fornecida pela B3, o índice do CDI em 29-12-2023 estava em **42805,60**, e em 31-01-2024 o índice estava em **43219,40**. Substituindo esses valores na fórmula, temos:

In [368]:
def calc_cdi_monthly_rate(start_index: float, end_index: float) -> float:
    return end_index / start_index - 1

jan_monthly_rate = calc_cdi_monthly_rate(start_index=42805.60, end_index=43219.40)

print(f"Taxa mensal: {jan_monthly_rate}")
print(f"Taxa mensal arredondada: {round(jan_monthly_rate, 4)}")


Taxa mensal: 0.009666959463247915
Taxa mensal arredondada: 0.0097


A taxa anual é calculada da mesma maneira, porém a taxa é anualizada. Se utiliza essa fórmula:

$$\huge(1 + Taxa_{mensal})^{\frac{252}{Ndu}} - 1$$

Onde Ndu é o número de dias úteis no mês calculado.

Criar uma função para calcular a taxa anualizada no python é um pouco mais complicado, porque o calendário de feriados que devemos utilizar é diferente do que o pandas utiliza na função bdate_rate. Mas, para demonstrar como esse cálculo seria, carregamos uma lista personalizada dos feriados de Janeiro de 2024 (retirada no [Calendário da FEBRABAM](https://feriadosbancarios.febraban.org.br/)), e utilizamos esse número na conta.

In [369]:
jan_holidays = ["2024-01-01"]

jan_bday_febraban = CustomBusinessDay(holidays=jan_holidays)

start = pd.Timestamp(2024, 1, 1)
end = start + pd.offsets.MonthEnd(0)

bdays_no = len(pd.date_range(start, end, freq=jan_bday_febraban))

def annualize_cdi(monthly_rate: float, bdays_no: int) -> float:
    return (1 + monthly_rate)**(252 / bdays_no) - 1

annalized_rate = annualize_cdi(monthly_rate=jan_monthly_rate, bdays_no=bdays_no)
print(f"Taxa anualizada: {annalized_rate}")
print(f"Taxa anualizada arredondada: {round(annalized_rate, 4)}")

Taxa anualizada: 0.11650004967146854
Taxa anualizada arredondada: 0.1165
0.11650004967146854


Perfeito, agora resta falar sobre os números calculados pela Pipeline. Em específico, o cdi_accumulated_ytd_rate e o cdi_12m_rate. Esses dois campos são preenchidos com a conta de acúmulos a seguir. A única diferença entre as duas é que o cdi_accumulated_ytd_rate é o acúmulo do começo do ano até o mês que está sendo calculado, enquanto cdi_12m_rate é o acúmulo dos últimos 12 meses, não importando se esse meses forem do mês anterior.

Cálculo de acúmulos:

$$\large\prod_{i=1}^{n} (1 + Taxa_{i}) - 1$$

Função em Python:

In [ ]:
# Função em src/transform/base_transform.py
def calc_accumulated_ytd_rate(monthly_rates: List[float]) -> float:
    factor = 1.0
    for rate in monthly_rates:
        factor *= (1.0 + rate)
    return factor - 1.0

### 2.1.2 IPCA
O Índice Nacional de Preços ao Consumidor Amplo (IPCA) é o índice oficial de inflação do Brasil. Ele é calculado mensalmente pelo IBGE, e reflete a variação de preços de um conjunto de bens e serviços consumidos pelas famílias brasileiras. Ele é relevante para o investidor comum porque a inflação tem um impacto direto no poder de compra do dinheiro.

Esse é o esquema de como o IPCA é armazenado.

In [370]:
# Você pode consultar ss esquemas das planilhas em src/storage/processed/schema.py
def ipca_schema() -> pd.DataFrame:
    df = pd.DataFrame({
        "date": [],
        "ipca_monthly_rate": [],
        "ipca_accumulated_ytd_rate": [],
        "ipca_12m_rate": [],
    })
    return df

Seus dados brutos também são armazenados de uma maneira semelhante ao do CDI.